<a href="https://colab.research.google.com/github/n5n0/eskwelabs-portfolio/blob/main/py6_clean_practice_bank_Crinan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# py6 · Clean: Python Practice Bank (Day 6)

**DABC Sprint 2 · AML case · the same five tables you queried in SQL on Days 1–4**

### How to use this bank

- Questions are tiered **EASY · AVERAGE · DIFFICULT**. Work in order.
- **In class:** Practice block 1 (18:45) → **Q1–12**, after you've standardised `occupation` in Notebook 2. Practice block 2 (19:45) → **Q13–25**. Your mentor will tell you the floor (what everyone finishes) for your room.
- **SQL twin** names the Day 1–4 question that asks the same thing. When both use the same table, your Python answer should give the same number as your SQL answer. If it doesn't, find out why: that's the lesson.
- Try each question yourself first. Then use AI to explain or check, following the sprint's AI protocol: paste the column names, never rows of data, and note it in your AI use log.
- ✍️ cells ask for a sentence, not code. Write it for the Head of AML.
- Your mentor has the answer key. Remaining questions are homework.

**Data you need** (Sprint 2 student folder, the same tables you queried in BigQuery on Days 1–4): `customers.csv`, `accounts.csv`, `transactions.csv`, `alerts.csv`, `investigations.csv` and `aml_analytical_extract.csv` (the Day 4 extract). Upload them to this Colab session, or put them in a Drive folder and add that folder to `DATA_DIRS` below.

## Step 0 · Setup

Run these four cells in order with **Shift + Enter**. After a runtime restart, run them again.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("pandas", pd.__version__, "· ready")

pandas 2.2.3 · ready


In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION — the only cell you should need to edit
# ---------------------------------------------------------------------------
USE_BIGQUERY = False                  # True = read the tables straight from BigQuery
PROJECT_ID   = "your-project-id"      # ← paste the project ID from your instructor
DATASET      = "aml_sprint2"

# Folders searched, in order, for the CSV files. Add your own Drive folder if needed.
DATA_DIRS = [
    ".",                                          # same folder as this notebook
    "data",
    "/content",                                   # Colab: files uploaded to this session
    "/content/drive/MyDrive/DABC24/aml_sprint2",  # Colab + Drive (mount Drive first)
]
print("USE_BIGQUERY =", USE_BIGQUERY)

USE_BIGQUERY = False


In [ ]:
def load(name):
    """Return one table as a DataFrame.

    name is one of: customers, accounts, transactions, alerts, investigations,
    aml_analytical_extract. Prints where the data came from, so you always know.
    """
    if USE_BIGQUERY:
        from google.colab import auth
        from google.cloud import bigquery
        auth.authenticate_user()
        client = bigquery.Client(project=PROJECT_ID)
        frame = client.query(
            f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{name}`"
        ).to_dataframe()
        print(f"{name:<24} {len(frame):>7,} rows  (BigQuery)")
        return frame

    for folder in DATA_DIRS:
        path = os.path.join(folder, f"{name}.csv")
        if os.path.exists(path):
            frame = pd.read_csv(path)
            print(f"{name:<24} {len(frame):>7,} rows  ({path})")
            return frame

    raise FileNotFoundError(
        f"{name}.csv not found in {DATA_DIRS}. Upload it to this session "
        "or add its folder to DATA_DIRS in the configuration cell."
    )

In [ ]:
raw = load("aml_analytical_extract")
df = raw.copy()            # work on a copy, so raw stays exactly as delivered
transactions = load("transactions")
accounts     = load("accounts")
customers    = load("customers")
alerts       = load("alerts")

aml_analytical_extract    46,793 rows  (./aml_analytical_extract.csv)
transactions              46,838 rows  (./transactions.csv)
accounts                  12,028 rows  (./accounts.csv)
customers                 10,000 rows  (./customers.csv)
alerts                     2,618 rows  (./alerts.csv)


## Block 1 · Text cleaning and duplicates (Q1–12)

Rule for the whole bank: **`raw` never changes.** Every cleaned column is a *new* column in `df`.

### Q1 · EASY · SQL twin: d3a Q5

How many distinct values does `df["channel"]` have, counting missing as one value?

💡 *Hint:* `.nunique(dropna=False)`.

In [ ]:
# Your code here
df["channel"].nunique(dropna=False)

22

### Q2 · EASY

Apply `.str.strip()` to `channel`. How many distinct non-missing values are left? Which spelling did stripping fix?

💡 *Hint:* Compare the set of values before and after: `set(a) - set(b)`.

In [ ]:
# Your code here
a = df["channel"].dropna().unique()
b = df["channel"].str.strip().dropna().unique()

print("Distinct values left after strip:", len(b))
print("Spelling fixed:", set(a) - set(b))

Distinct values left after strip: 19
Spelling fixed: {'Branch ', ' Online'}


### Q3 · EASY

Now strip **and** lowercase `channel`. How many distinct non-missing values are left? List them.

In [ ]:
# Q3: Strip and lowercase the channel column
channel_cleaned_series = df["channel"].str.strip().str.lower().dropna()
distinct_vals = channel_cleaned_series.unique()

print("Number of distinct non-missing values left:", len(distinct_vals))
print("Distinct values:", sorted(distinct_vals))

Number of distinct non-missing values left: 6
Distinct values: ['atm', 'branch', 'instapay', 'mobile', 'online', 'pesonet']


### Q4 · AVERAGE

Create a new column `channel_clean`: strip, lowercase, then `.map()` with a dictionary to the six labels the bank uses: `ATM`, `Branch`, `InstaPay`, `Mobile`, `Online`, `PESONet`. Show `value_counts(dropna=False)`.

💡 *Hint:* `CHANNEL_MAP = {"atm": "ATM", ...}` then `df["channel"].str.strip().str.lower().map(CHANNEL_MAP)`.

In [ ]:
# Q4: Create channel_clean column and display value counts
CHANNEL_MAP = {
    "atm": "ATM",
    "branch": "Branch",
    "instapay": "InstaPay",
    "mobile": "Mobile",
    "online": "Online",
    "pesonet": "PESONet"
}

df["channel_clean"] = df["channel"].str.strip().str.lower().map(CHANNEL_MAP)
display(df["channel_clean"].value_counts(dropna=False))

,count
channel_clean,
Online,8502
Branch,8477
PESONet,8014
Mobile,7233
InstaPay,7228
ATM,7199
NaN,140


### Q5 · AVERAGE

Prove `channel_clean` changed nothing it shouldn't have. Write three `assert` statements: same number of rows as `raw`; same `amount` total; and the same number of missing values as the raw `channel` (so `.map()` didn't create new ones).

💡 *Hint:* `assert condition, "message if it fails"`. Compare totals with `round(..., 2)`.

In [ ]:
# Q5: Assert statements to verify channel_clean
# 1. Same number of rows as raw
assert len(df) == len(raw), f"Row count mismatch: {len(df)} vs {len(raw)}"

# 2. Same amount total (comparing rounded values to avoid float precision issues)
assert round(df["amount"].sum(), 2) == round(raw["amount"].sum(), 2), "Total amount mismatch"

# 3. Same number of missing values as the raw channel
assert df["channel_clean"].isna().sum() == raw["channel"].isna().sum(), "Missing values count mismatch"

print("All assertions passed successfully!")

All assertions passed successfully!


### Q6 · EASY

How many distinct `occupation` labels are in `df` as delivered? How many after `.str.strip().str.lower()`? List the lowercased values with their counts.

In [ ]:
# Q6: Clean and inspect occupation column
raw_unique_count = df["occupation"].nunique(dropna=False)

cleaned_occupation = df["occupation"].str.strip().str.lower()
cleaned_unique_count = cleaned_occupation.nunique(dropna=False)

print(f"Distinct occupation labels in df as delivered: {raw_unique_count}")
print(f"Distinct labels after .str.strip().str.lower(): {cleaned_unique_count}\n")
print("Lowercased values with their counts:")
display(cleaned_occupation.value_counts(dropna=False))

Distinct occupation labels in df as delivered: 19
Distinct labels after .str.strip().str.lower(): 7

Lowercased values with their counts:


,count
occupation,
professional,21612
business owner,14265
student,4328
unemployed,4092
NaN,1773
self-employed,525
not employed,198


### Q7 · AVERAGE

The bank's onboarding form has four occupations: **Professional, Business Owner, Student, Unemployed**. Decide where `self-employed` and `not employed` belong and write your decision in the ✍️ cell. Then create `occupation_clean` with `.map()` and show its counts.

In [ ]:
# Q7: Clean and map occupation labels
OCCUPATION_MAP = {
    "professional": "Professional",
    "business owner": "Business Owner",
    "self-employed": "Business Owner",
    "student": "Student",
    "unemployed": "Unemployed",
    "not employed": "Unemployed"
}

df["occupation_clean"] = df["occupation"].str.strip().str.lower().map(OCCUPATION_MAP)
display(df["occupation_clean"].value_counts(dropna=False))

,count
occupation_clean,
Professional,21612
Business Owner,14790
Student,4328
Unemployed,4290
NaN,1773


✍️ **My decision for `self-employed` and `not employed`, and why:**

*Your answer:*

### ✍️ Decision on Mapping Occupations
* **`self-employed`**: Mapped to **`Business Owner`** because they run their own business or independent practice.
* **`not employed`**: Mapped to **`Unemployed`** since they are currently without employment.

### Q8 · AVERAGE

`.map()` turns any value that isn't in your dictionary into NaN, silently. Check that `occupation_clean` has exactly as many missing values as the raw `occupation`. If it doesn't, which values fell through?

💡 *Hint:* Fell through = missing *after* but not missing *before*: `df["occupation_clean"].isna() & df["occupation"].notna()`.

In [ ]:
# Q8: Check for any values that fell through the mapping
fell_through_mask = df["occupation_clean"].isna() & df["occupation"].notna()
fell_through_values = df.loc[fell_through_mask, "occupation"].unique()

print("Number of missing values in raw occupation:", df["occupation"].isna().sum())
print("Number of missing values in occupation_clean:", df["occupation_clean"].isna().sum())
print("Did any values fall through?", "Yes" if len(fell_through_values) > 0 else "No")
if len(fell_through_values) > 0:
    print("Values that fell through:", fell_through_values)


Number of missing values in raw occupation: 1773
Number of missing values in occupation_clean: 1773
Did any values fall through? No


### Q9 · EASY · SQL twin: Day 1 Activity 2, stretch S1

In the raw `transactions` table: how many rows are **exact** duplicates of another row? How many rows have a **duplicated `tx_id`**?

💡 *Hint:* `.duplicated().sum()` checks whole rows; `.duplicated(subset="tx_id").sum()` checks the key.

In [ ]:
# Q9: Calculate exact row duplicates and duplicated transaction keys (tx_id)
exact_row_duplicates = transactions.duplicated().sum()
tx_id_duplicates = transactions.duplicated(subset="tx_id").sum()

print(f"Number of exact duplicate rows: {exact_row_duplicates}")
print(f"Number of rows with a duplicated tx_id: {tx_id_duplicates}")


Number of exact duplicate rows: 33
Number of rows with a duplicated tx_id: 45


### Q10 · DIFFICULT

The two numbers in Q9 differ. Find the `tx_id`s that are duplicated but **not** exact duplicates, and show which column differs between the copies.

💡 *Hint:* Keep all copies with `.duplicated(subset="tx_id", keep=False)`. Then `groupby("tx_id").nunique() > 1` shows which columns vary inside a group.

In [ ]:
# Q10: Find tx_ids that are duplicated but not exact duplicates, and identify differing columns
# 1. Get all copies of duplicated tx_ids
all_dup_tx = transactions[transactions.duplicated(subset="tx_id", keep=False)]

# 2. Group by tx_id and find which columns have more than 1 unique value inside the group
unique_counts_per_group = all_dup_tx.groupby("tx_id").nunique()
differing_cols_mask = unique_counts_per_group > 1

# Filter to find groups (tx_ids) that actually have differing columns (i.e. not exact duplicates)
non_exact_dups = unique_counts_per_group[differing_cols_mask.any(axis=1)]

print(f"Number of tx_ids that are duplicated but not exact duplicates: {len(non_exact_dups)}")
print("\nColumns that differ within these groups and their number of unique values:")
display(non_exact_dups.loc[:, (non_exact_dups > 1).any(axis=0)])


Number of tx_ids that are duplicated but not exact duplicates: 12

Columns that differ within these groups and their number of unique values:


,channel
tx_id,
TX-059fe9b255,2
TX-13c72aa212,2
TX-1ac5d76a10,2
TX-274cd00dba,2
TX-4caa7cbc5a,2
TX-524327f24c,2
TX-65cbf9079f,2
TX-68dfde526e,2
TX-a929a50922,2


✍️ **What does this tell you about the order of your cleaning steps?**

*Your answer:*

### Q11 · AVERAGE · SQL twin: 04_final_analytical_extract.sql, transactions_dedup

De-duplicate the raw `transactions` on `tx_id`, keeping the earliest `tx_datetime`, and call the result `transactions_d`. Confirm it has 46,793 rows and an amount total of PHP 3,250,542,708.32, the same as the extract.

💡 *Hint:* `.sort_values("tx_datetime", kind="stable")` then `.drop_duplicates(subset="tx_id", keep="first")`. This is the pandas version of the SQL `QUALIFY ROW_NUMBER() … = 1`.

In [ ]:
# Q11: De-duplicate transactions keeping the earliest record
transactions_d = (
    transactions
    .sort_values("tx_datetime", kind="stable")
    .drop_duplicates(subset="tx_id", keep="first")
)

# Confirm row count and total transaction amount
row_count = len(transactions_d)
total_amount = transactions_d["amount"].sum()

print(f"Row count: {row_count:,} (Expected: 46,793)")
print(f"Total Amount: PHP {total_amount:,.2f} (Expected: PHP 3,250,542,708.32)")

# Verify against expected values
assert row_count == 46793, f"Row count mismatch: {row_count}"
assert round(total_amount, 2) == 3250542708.32, f"Amount mismatch: {total_amount}"
print("Confirmation successful!")

Row count: 46,793 (Expected: 46,793)
Total Amount: PHP 3,250,542,708.32 (Expected: PHP 3,250,542,708.32)
Confirmation successful!


### Q12 · DIFFICULT

`accounts` has 28 duplicated `acct_id`s, but `accounts.duplicated().sum()` finds only 27 exact duplicate rows. Find the odd one out. What is different, and is its owner in `customers`?

💡 *Hint:* Same technique as Q10, on `accounts`.

In [ ]:
# Q12: Find the odd one out among duplicated accounts
# 1. Get all copies of duplicated acct_ids
all_dup_accts = accounts[accounts.duplicated(subset="acct_id", keep=False)]

# 2. Group by acct_id and check which columns differ
unique_counts = all_dup_accts.groupby("acct_id").nunique()
differing_cols_mask = unique_counts > 1
non_exact_dup_accts = unique_counts[differing_cols_mask.any(axis=1)]

print(f"Number of acct_ids with differing details: {len(non_exact_dup_accts)}")
print("Columns that differ:")
display(non_exact_dup_accts.loc[:, (non_exact_dup_accts > 1).any(axis=0)])

# Show the actual rows for the odd one out
odd_acct_id = non_exact_dup_accts.index[0]
odd_acct_rows = all_dup_accts[all_dup_accts["acct_id"] == odd_acct_id]
print(f"\nDetails of the odd account ({odd_acct_id}):")
display(odd_acct_rows)

# 3. Check if its owners are in the customers table
owner_ids = odd_acct_rows["cust_id"].unique()
for owner_id in owner_ids:
    exists = owner_id in customers["cust_id"].values
    print(f"Is owner '{owner_id}' present in customers table? {exists}")

Number of acct_ids with differing details: 1
Columns that differ:


,cust_id
acct_id,
ACC-db44b19b,2



Details of the odd account (ACC-db44b19b):


,acct_id,cust_id,acct_type,open_date,initial_balance,acct_status
6490,ACC-db44b19b,CUST-799174f8,Checking,2021-01-17,"4,336.87",Active
12027,ACC-db44b19b,CUST-ceaef578,Checking,2021-01-17,"4,336.87",Active


Is owner 'CUST-799174f8' present in customers table? False
Is owner 'CUST-ceaef578' present in customers table? True


✍️ **What would you write in the cleaning log?**

*Your answer:*

## Block 2 · Masks, types, rules and flags (Q13–25)

### ✅ Checkpoint

Run the next cell before Q13.

In [ ]:
# CHECKPOINT: run this before Q13.
# It repeats the model answers to Q4, Q7 and Q11, so the rest of the bank works
# even if you skipped them. It overwrites your versions with the model decisions.
CHANNEL_MAP = {"atm": "ATM", "branch": "Branch", "instapay": "InstaPay",
               "mobile": "Mobile", "online": "Online", "pesonet": "PESONet"}
OCCUPATION_MAP = {"professional": "Professional", "business owner": "Business Owner",
                  "self-employed": "Business Owner", "student": "Student",
                  "unemployed": "Unemployed", "not employed": "Unemployed"}
df["channel_clean"] = df["channel"].str.strip().str.lower().map(CHANNEL_MAP)
df["occupation_clean"] = df["occupation"].str.strip().str.lower().map(OCCUPATION_MAP)
transactions_d = (transactions
                  .sort_values("tx_datetime", kind="stable")
                  .drop_duplicates(subset="tx_id", keep="first"))
print("checkpoint ready:", df.shape, f"transactions_d: {len(transactions_d):,} rows")

checkpoint ready: (46793, 51) transactions_d: 46,793 rows


### Q13 · EASY

Convert `alerts["alert_score"]` to numbers in a new column `alert_score_num` with `pd.to_numeric(..., errors="coerce")`. How many values became NaN, and what were they originally?

In [ ]:
# Q13: Convert alert_score to numeric and analyze values that became NaN
# Convert to numeric, forcing errors to NaN
alerts["alert_score_num"] = pd.to_numeric(alerts["alert_score"], errors="coerce")

# Identify indices where conversion resulted in NaN but the original value was not NaN
nan_mask = alerts["alert_score_num"].isna() & alerts["alert_score"].notna()
nan_count = nan_mask.sum()
original_nan_values = alerts.loc[nan_mask, "alert_score"].unique()

print(f"Number of values that became NaN: {nan_count}")
if nan_count > 0:
    print("Original non-numeric values:", original_nan_values)


Number of values that became NaN: 157
Original non-numeric values: ['NOT_SCORED']


### Q14 · EASY

Convert `customers["customer_since"]` to datetimes. What are the earliest and latest dates?

In [ ]:
# Q14: Convert customer_since to datetime and find the date range
customers["customer_since_dt"] = pd.to_datetime(customers["customer_since"])

earliest_date = customers["customer_since_dt"].min()
latest_date = customers["customer_since_dt"].max()

print(f"Earliest customer sign-up date: {earliest_date.strftime('%Y-%m-%d')}")
print(f"Latest customer sign-up date:  {latest_date.strftime('%Y-%m-%d')}")

Earliest customer sign-up date: 2015-01-01
Latest customer sign-up date:  2024-12-28


### Q15 · AVERAGE · SQL twin: d2 Q12

How many transactions are worth at least PHP 500,000 **and** were made online? Count it twice: once with the raw `channel == "Online"` and once with `channel_clean == "Online"`. How many did the raw version miss?

💡 *Hint:* Combine conditions with `&`, and put each condition in its own brackets.

In [ ]:
# Q15: Count transactions >= 500,000 and made online using raw vs clean channel

# Count using raw channel
raw_online_count = len(df[(df["amount"] >= 500000) & (df["channel"] == "Online")])

# Count using cleaned channel
clean_online_count = len(df[(df["amount"] >= 500000) & (df["channel_clean"] == "Online")])

# Calculate how many were missed by the raw version
missed_count = clean_online_count - raw_online_count

print(f"Count using raw 'channel': {raw_online_count:,}")
print(f"Count using 'channel_clean': {clean_online_count:,}")
print(f"Transactions missed by raw version: {missed_count:,}")

Count using raw 'channel': 417
Count using 'channel_clean': 484
Transactions missed by raw version: 67


### Q16 · AVERAGE · SQL twin: d2 Q10

Use `.isin()` to count transactions made through **Online, Mobile or ATM**, using `channel_clean`.

In [ ]:
# Q16: Count transactions made through Online, Mobile, or ATM using channel_clean
channels_of_interest = ["Online", "Mobile", "ATM"]
matching_transactions_count = df["channel_clean"].isin(channels_of_interest).sum()

print(f"Number of transactions made through Online, Mobile, or ATM: {matching_transactions_count:,}")

Number of transactions made through Online, Mobile, or ATM: 22,934


### Q17 · AVERAGE · SQL twin: d2 Q11

Count the near-threshold transactions: at least PHP 450,000 **and** below PHP 500,000. Check that your mask agrees with the extract's `near_threshold_flag` on every row.

💡 *Hint:* Compare two columns row by row and finish with `.all()`. Convert True/False to 1/0 with `.astype(int)`.

In [ ]:
# Q17: Count near-threshold transactions and verify against near_threshold_flag
near_threshold_mask = (df["amount"] >= 450000) & (df["amount"] < 500000)
near_threshold_count = near_threshold_mask.sum()

# Convert boolean mask to 1/0 and check if it matches the extract's flag on all rows
mask_matches_extract = (near_threshold_mask.astype(int) == df["near_threshold_flag"]).all()

print(f"Number of near-threshold transactions: {near_threshold_count:,}")
print(f"Does the mask match the extract's flag on every row? {mask_matches_extract}")

Number of near-threshold transactions: 865
Does the mask match the extract's flag on every row? True


### Q18 · AVERAGE · SQL twin: d2 Q14

Find the alerts whose `rule_name` contains *structuring*, ignoring case. How many are there?

In [ ]:
# Q18: Find alerts containing "structuring" in rule_name (case-insensitive)
structuring_alerts = alerts[alerts["rule_name"].str.contains("structuring", case=False, na=False)]
structuring_count = len(structuring_alerts)

print(f"Number of alerts containing 'structuring': {structuring_count}")

### Q19 · EASY

Use `np.where` to create `is_external`: 1 when `dest_acct` starts with `EXT`, otherwise 0. Does it agree with the extract's `dest_is_internal` on every row? How many external transactions are there?

In [ ]:
# Q19: Create is_external using np.where and verify against dest_is_internal
df["is_external"] = np.where(df["dest_acct"].str.startswith("EXT"), 1, 0)

# Count external transactions
external_count = (df["is_external"] == 1).sum()

# Verification: since 'external' is the opposite of 'internal',
# we check if is_external is always the opposite of dest_is_internal (i.e. is_external + dest_is_internal == 1)
agrees_with_extract = (df["is_external"] == (1 - df["dest_is_internal"])).all()

print(f"Number of external transactions: {external_count:,}")
print(f"Does the is_external flag agree with the extract's dest_is_internal on every row? {agrees_with_extract}")

Number of external transactions: 32,471
Does the is_external flag agree with the extract's dest_is_internal on every row? True


### Q20 · AVERAGE · SQL twin: d2 Q18

Use `np.select` to label each transaction `Large` (≥ 500,000), `Near Threshold` (450,000 to below 500,000) or `Below Threshold Range`. Store it in `amount_class` and show the counts.

💡 *Hint:* `np.select([condition1, condition2], ["Large", "Near Threshold"], default="Below Threshold Range")`. The first condition that is True wins, just like `CASE WHEN`.

In [ ]:
# Q20: Categorize transactions using np.select
conditions = [
    df["amount"] >= 500000,
    (df["amount"] >= 450000) & (df["amount"] < 500000)
]

choices = ["Large", "Near Threshold"]

df["amount_class"] = np.select(conditions, choices, default="Below Threshold Range")

# Display the value counts of the newly created column
display(df["amount_class"].value_counts(dropna=False))

,count
amount_class,
Below Threshold Range,44388
Large,1540
Near Threshold,865


### Q21 · DIFFICULT

Recreate the extract's `amount_band` column (five bands: see the `CASE` in `04_final_analytical_extract.sql`) with `np.select`, as `amount_band_py`. Prove it matches the SQL column on every row.

💡 *Hint:* Bands: `invalid_negative` (< 0), `under_50k`, `50k_to_450k`, `near_threshold_450k_500k`, and the default `at_or_above_500k`. Because the first True condition wins, each condition only needs an upper bound.

In [ ]:
# Q21: Recreate amount_band column using np.select
conditions = [
    df["amount"] < 0,                      # invalid_negative
    df["amount"] < 50000,                  # under_50k
    df["amount"] < 450000,                 # 50k_to_450k
    df["amount"] < 500000                  # near_threshold_450k_500k
]

choices = [
    "invalid_negative",
    "under_50k",
    "50k_to_450k",
    "near_threshold_450k_500k"
]

df["amount_band_py"] = np.select(conditions, choices, default="at_or_above_500k")

# Check if it matches the original SQL extract's column 'amount_band' on every row
matches_perfectly = (df["amount_band_py"] == df["amount_band"]).all()
print(f"Does the generated amount_band_py match the extract's amount_band on every row? {matches_perfectly}")

# Display some value counts to verify
display(df["amount_band_py"].value_counts(dropna=False))

Does the generated amount_band_py match the extract's amount_band on every row? True


,count
amount_band_py,
under_50k,40413
50k_to_450k,3953
at_or_above_500k,1540
near_threshold_450k_500k,865
invalid_negative,22


### Q22 · AVERAGE

Flag, don't delete. In `customers`, create `age_flag` with values `missing`, `implausible` (age over 100) or `ok`. How many customers are in each group? Then: how many **transactions** in `df` come from customers with an implausible age?

In [ ]:
# Q22: Categorize customer ages and count corresponding transactions
import numpy as np

# 1. Create age_flag in customers
conditions = [
    customers["age"].isna(),
    customers["age"] > 100
]
choices = ["missing", "implausible"]
customers["age_flag"] = np.select(conditions, choices, default="ok")

# Display how many customers are in each group
print("Customer counts by age group:")
print(customers["age_flag"].value_counts(dropna=False))

# 2. Find how many transactions in df come from customers with an implausible age
# First de-duplicate accounts to have a unique mapping index
unique_accounts = accounts.drop_duplicates(subset="acct_id")
acct_to_cust_map = unique_accounts.set_index("acct_id")["cust_id"]

# Map each transaction's originating account in df to its owner's cust_id
df["cust_id"] = df["orig_acct"].map(acct_to_cust_map)

# Identify the cust_ids of customers with implausible age
implausible_cust_ids = customers.loc[customers["age_flag"] == "implausible", "cust_id"]

# Filter df for transactions belonging to those customers
implausible_tx_count = df["cust_id"].isin(implausible_cust_ids).sum()
print(f"\nNumber of transactions from customers with an implausible age: {implausible_tx_count:,}")

Customer counts by age group:
age_flag
ok             9986
implausible      14
Name: count, dtype: int64

Number of transactions from customers with an implausible age: 54


### Q23 · AVERAGE

How many negative amounts are in `df`, and what do they add up to? Check your mask against the extract's `negative_amount_flag`.

In [ ]:
# Q23: Count and sum negative amounts, and verify against negative_amount_flag
negative_mask = df["amount"] < 0
negative_count = negative_mask.sum()
negative_sum = df.loc[negative_mask, "amount"].sum()

# Check if the mask matches negative_amount_flag on every row
matches_extract = (negative_mask.astype(int) == df["negative_amount_flag"]).all()

print(f"Number of negative amounts in df: {negative_count}")
print(f"Total sum of negative amounts: PHP {negative_sum:,.2f}")
print(f"Does our mask agree with 'negative_amount_flag' on every row? {matches_extract}")

Number of negative amounts in df: 22
Total sum of negative amounts: PHP -212,638.25
Does our mask agree with 'negative_amount_flag' on every row? True


✍️ **Should these rows be deleted? What would you write in the cleaning log?**

*Your answer:*

### Q24 · DIFFICULT

Recompute `amount_to_income_ratio` in pandas as `ratio_py`, so it is NaN when `declared_income` is 0 or missing. Prove it matches the extract's column on every row.

💡 *Hint:* `income.where(income > 0)` turns 0 and missing into NaN, so the division can't blow up. Compare with `np.isclose(a, b, equal_nan=True)`.

In [ ]:
# Q24: Recompute amount_to_income_ratio as ratio_py
# Replace 0 or missing incomes with NaN to avoid division by zero or invalid calculations
safe_income = df["declared_income"].where(df["declared_income"] > 0)

df["ratio_py"] = df["amount"] / safe_income

# Prove it matches the extract's amount_to_income_ratio on every row
# np.isclose is used to safely handle floating point precision and NaNs (via equal_nan=True)
matches_perfectly = np.isclose(df["ratio_py"], df["amount_to_income_ratio"], equal_nan=True).all()

print(f"Does our recomputed ratio_py match the extract's amount_to_income_ratio on every row? {matches_perfectly}")

# Display some sample values for confirmation
display(df[["amount", "declared_income", "amount_to_income_ratio", "ratio_py"]].head())


Does our recomputed ratio_py match the extract's amount_to_income_ratio on every row? True


,amount,declared_income,amount_to_income_ratio,ratio_py
0,343.32,"375,856.18",0.00,0.00
1,198.66,"925,897.02",0.00,0.00
2,625.91,"1,281,739.05",0.00,0.00
3,"2,191.55","179,380.69",0.01,0.01
4,45.14,"45,334.44",0.00,0.00


### Q25 · DIFFICULT

Write a function `control_totals(frame)` that returns a dict: rows, distinct `tx_id`, total amount (2 decimal places), total alerts, total investigations. Show `raw` and `df` side by side in one DataFrame, with a `match` column.

In [ ]:
# Q25: Define control_totals function and display side-by-side comparison
def control_totals(frame):
    """Return a dictionary of key control totals for a given DataFrame."""
    # 1. Total rows
    rows = len(frame)

    # 2. Distinct tx_id
    distinct_tx = frame["tx_id"].nunique()

    # 3. Total amount (rounded to 2 decimal places)
    total_amount = round(frame["amount"].sum(), 2)

    # 4. Total alerts (sum of alert counts)
    total_alerts = int(frame["alert_count"].sum()) if "alert_count" in frame.columns else 0

    # 5. Total investigations (sum of investigation counts)
    total_investigations = int(frame["was_investigated"].sum()) if "was_investigated" in frame.columns else 0

    return {
        "rows": rows,
        "distinct_tx_id": distinct_tx,
        "total_amount": total_amount,
        "total_alerts": total_alerts,
        "total_investigations": total_investigations
    }

# Calculate totals for raw and df
raw_totals = control_totals(raw)
df_totals = control_totals(df)

# Create a side-by-side comparison DataFrame
comparison_df = pd.DataFrame({
    "raw": pd.Series(raw_totals),
    "df": pd.Series(df_totals)
})

# Add a boolean column indicating if they match
comparison_df["match"] = comparison_df["raw"] == comparison_df["df"]

# Display the final comparison table
display(comparison_df)

,raw,df,match
rows,"46,793.00","46,793.00",True
distinct_tx_id,"46,793.00","46,793.00",True
total_amount,"3,250,542,708.32","3,250,542,708.32",True
total_alerts,"2,618.00","2,618.00",True
total_investigations,"1,537.00","1,537.00",True


---
### Done?

Unfinished questions are homework. Bring the one you were most stuck on to tomorrow's opener, along with what you tried.